### L shaped antenna simulation.

In this notebook we will generate the mesh of an L shaped antenna and generate a json for a transient
simulation.

In [45]:
import gmsh
import math
import os
import sys
import json

from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent))
from palace.viz import view_mesh
from palace.geometry import extract_tag, xmin, xmax, ymin, ymax, zmin, zmax
from palace.mesh import refine_near_surfaces, refine_surfaces, run_boolean_pipeline, Entity


### Parameters:
- h : Patch height along z-axis, specified as a scalar in meters. 
- l1 : Ground plane length along x-axis, specified as a scalar in meters
- w1 : Ground plane width along y-axis, specified as a scalar in meters
- w3 : Strip line width along y-axis, specified as a scalar in meters.
- air_height : Air box height along z-axis, specified as a scalar in meters.  
- air_margin : Air box margin along x and y axes, specified as a scalar in meters.
- freq  : Simulation frequency in GHz, specified as a scalar.
- filename : Output mesh filename, specified as a string.

In [46]:
l1: float = 0.06
w1: float = 0.06
w3: float = 0.002
h: float = 0.0013
air_height: float = 0.025    
air_margin: float = 0.025    
freq: float = 3.3
filename: str = "l_antenna.msh"


eps_r: float = 2.2
loss_tan: float = 0.0009

### Initialize the model

In [47]:
gmsh.initialize()
gmsh.model.add("patch_antenna")
kernel = gmsh.model.occ

### Geometry Construction


In [48]:
total_xmin = -l1/2 - air_margin
total_xmax = l1/2 + air_margin
total_ymin = -w1/2 - air_margin
total_ymax = w1/2 + air_margin
total_zmax = h + air_height


# 1. Create volumes
substrate = kernel.addBox(-l1/2, -w1/2, 0, l1, w1, h)
air_box = kernel.addBox(total_xmin, total_ymin, 0,
                        total_xmax - total_xmin,
                        total_ymax - total_ymin,
                        total_zmax)
kernel.synchronize()


# 2. Fragment volumes FIRST (substrate carved out of air)
vol_result, vol_map = kernel.fragment(
    [(3, air_box)],
    [(3, substrate)]
)
kernel.synchronize()

# Track new tags from fragment
# vol_map[0] = what air_box became (may be multiple pieces)
# vol_map[1] = what substrate became
air_pieces = vol_map[0]       # Air volume(s) after carving
sub_pieces = vol_map[1]       # Substrate volume(s)

print(f"Air volumes after fragment:       {air_pieces}")
print(f"Substrate volumes after fragment: {sub_pieces}")

# 3. Create 2D surfaces (ground, patch, ports)
ground_plane = kernel.addRectangle(-l1/2, -w1/2, 0, l1, w1)

strip_length_x = l1/2
strip_length_y = w1/2 + w3/2

feed_line_x = kernel.addRectangle(-l1/2, -w3/2, h, strip_length_x, w3)
feed_line_y = kernel.addRectangle(0, -w3/2, h, w3, strip_length_y)

top_conductor, top_map = kernel.fuse(
    [(2, feed_line_x)], [(2, feed_line_y)],
    removeObject=True, removeTool=True
)
kernel.synchronize()

# Lumped ports — build directly in-plane, no rotation
# Port 1: at x = -l1/2, vertical face
p1 = kernel.addPoint(-l1/2, -w3/2, 0)
p2 = kernel.addPoint(-l1/2,  w3/2, 0)
p3 = kernel.addPoint(-l1/2,  w3/2, h)
p4 = kernel.addPoint(-l1/2, -w3/2, h)

lp1_a = kernel.addLine(p1, p2)
lp1_b = kernel.addLine(p2, p3)
lp1_c = kernel.addLine(p3, p4)
lp1_d = kernel.addLine(p4, p1)
loop1 = kernel.addCurveLoop([lp1_a, lp1_b, lp1_c, lp1_d])
lumped_port_1 = kernel.addPlaneSurface([loop1])
kernel.synchronize()

# Port 2: at y = w1/2, vertical face
p5 = kernel.addPoint(0,  w1/2, 0)
p6 = kernel.addPoint(w3, w1/2, 0)
p7 = kernel.addPoint(w3, w1/2, h)
p8 = kernel.addPoint(0,  w1/2, h)

lp2_a = kernel.addLine(p5, p6)
lp2_b = kernel.addLine(p6, p7)
lp2_c = kernel.addLine(p7, p8)
lp2_d = kernel.addLine(p8, p5)
loop2 = kernel.addCurveLoop([lp2_a, lp2_b, lp2_c, lp2_d])
lumped_port_2 = kernel.addPlaneSurface([loop2])
kernel.synchronize()


# 4. Fragment: embed all surfaces into volumes
all_volumes = [e for e in air_pieces + sub_pieces]
all_surfaces = (
    [(2, ground_plane)]
    + top_conductor
    + [(2, lumped_port_1), (2, lumped_port_2)]
)

result, result_map = kernel.fragment(
    all_volumes,
    all_surfaces
)
kernel.synchronize()

Air volumes after fragment:       [(3, 2), (3, 1)]
Substrate volumes after fragment: [(3, 1)]


### Defining Physical Groups
We categorize the geometry parts by checking their coordinates. This automatically groups the antenna components (patch, ground, port) and surrounding regions (substrate, air) into "Physical Groups," which are essential for defining simulation boundaries and material properties in Palace.

In [49]:
# regions
all_2d = gmsh.model.getEntities(2)
all_3d = gmsh.model.getEntities(3)

tol = 1e-6

ground_tags = []
sides = []
top_conductor_tags = []
port1_tags = []
port2_tags = []
farfield_tags = []
substrate_tags = []
air_tags = []

# 2D surfaces
for dim, tag in all_2d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    xmin, ymin, zmin = bbox[0], bbox[1], bbox[2]
    xmax, ymax, zmax = bbox[3], bbox[4], bbox[5]
    
    # Ground plane
    if (math.isclose(zmin, 0, abs_tol=tol) and 
        math.isclose(zmax, 0, abs_tol=tol) and
        xmin >= -l1/2 - tol and xmax <= l1/2 + tol):
        ground_tags.append(tag)
        
    # top conductor
    elif (math.isclose(zmin, h, abs_tol=tol) and 
            math.isclose(zmax, h, abs_tol=tol) and
            math.isclose(xmin, -l1/2, abs_tol=tol) and
            math.isclose(ymax, w1/2, abs_tol=tol) and
            math.isclose(ymin, -w3/2, abs_tol=tol)):
        top_conductor_tags.append(tag)
        
    # Lumped port 1
    elif (math.isclose(xmin, -l1/2, abs_tol=tol) and 
            math.isclose(xmax, -l1/2, abs_tol=tol) and
            math.isclose(ymin, -w3/2, abs_tol=tol) and
            math.isclose(ymax, w3/2, abs_tol=tol)):
        port1_tags.append(tag)
        
    # Lumped port 2
    elif (math.isclose(zmin, 0, abs_tol=tol) and    
            math.isclose(zmax, h, abs_tol=tol) and
            math.isclose(ymin, w1/2, abs_tol=tol) and
            math.isclose(ymax, w1/2, abs_tol=tol) and
            math.isclose(xmin, 0, abs_tol=tol)):
        port2_tags.append(tag)
    
    # Far-field (outer air box surfaces)
    elif (math.isclose(xmin, -l1/2 - air_margin, abs_tol=tol) or
            math.isclose(xmax, l1/2 + air_margin, abs_tol=tol) or
            math.isclose(ymin, -w1/2 - air_margin, abs_tol=tol) or
            math.isclose(ymax, w1/2 + air_margin, abs_tol=tol) or
            math.isclose(zmax, h + air_height, abs_tol=tol)):
        farfield_tags.append(tag)
    else:
        sides.append(tag)

# 3D volumes
for dim, tag in all_3d:
    bbox = gmsh.model.getBoundingBox(dim, tag)
    zmax = bbox[5]
    
    if math.isclose(zmax, h, abs_tol=tol):
        substrate_tags.append(tag)
    else:
        air_tags.append(tag)

# physical groups
pg_map = {
}

if substrate_tags:
    pg_map["substrate"] = gmsh.model.addPhysicalGroup(3, substrate_tags, name = "Substrate")
if air_tags:
    pg_map["air"] = gmsh.model.addPhysicalGroup(3, air_tags, name = "Air")
if ground_tags:
    pg_map["ground_plane"] = gmsh.model.addPhysicalGroup(2, ground_tags, name = "GroundPlane")
if top_conductor:
    pg_map["patch"] = gmsh.model.addPhysicalGroup(2, top_conductor_tags, name = "Patch")
if port1_tags:
    pg_map["lumped_port_1"] = gmsh.model.addPhysicalGroup(2, port1_tags, name = "LumpedPort1")
if port2_tags:
    pg_map["lumped_port_2"] = gmsh.model.addPhysicalGroup(2, port2_tags, name = "LumpedPort2")
if farfield_tags:
    pg_map["farfield"] = gmsh.model.addPhysicalGroup(2, farfield_tags, name = "FarField")
if sides:
    pg_map["sides"] = gmsh.model.addPhysicalGroup(2, sides, name = "Sides")

### Mesh generation and refinement

In [50]:
wavelength = 3e8 / (freq * 1e9)
mesh_size = wavelength / 15

refine_surfaces(port1_tags + port2_tags, mesh_size) 

gmsh.model.mesh.generate(3)
gmsh.model.mesh.setOrder(1)

gmsh.option.setNumber("Mesh.MshFileVersion", 2.2)
gmsh.option.setNumber("Mesh.Binary", 0)
gmsh.write(filename)

gmsh.fltk.run()
gmsh.finalize()

### Simulation Configuration
We define the key parameters for the electromagnetic simulation here. These settings control the frequency sweep range, material properties (dielectric constant and loss tangent for the substrate), and solver-specific configurations like port impedance and mesh order.

- output_file  : output filename for the configuration JSON file
- freq : frequency for the simulation (GHz)  
- eps_r: relative permittivity of the substrate
- loss_tan: loss tangent of the substrate
- port_impedance: characteristic impedance of the lumped port (Ohms)
- solver_order: order of the finite element basis functions for the simulation (e.g., 1 for linear, 2 for quadratic)

In [51]:
output_file_transient: str = "l_antenna_transient.json"
output_file_driven: str = "l_antenna_driven.json"

freq: float = 3.16
eps_r: float = 2.2
loss_tan: float = 0.0009
port_impedance: float = 50.0
solver_order: int = 2


import numpy as np
eps_0 = 8.8541878128e-12
sigma = 2 * np.pi * freq * eps_0 * eps_r * loss_tan

### Generating the Palace Configuration File
Finally, we assemble the simulation parameters into two JSON configuration. One for a transient simulation and one for a driven where we check the s-parameters.

In [52]:
def attr(name):
        return [pg_map[name]] if name in pg_map else []

config = {
    "Problem": {
        "Type": "Transient",
        "Verbose": 2,
        "Output": "/work/results_transient/l_antenna/"
    },

    "Model": {
        "Mesh": f"/work/{filename}",
        "L0": 1.0,
        "Refinement": {}
    },

    "Domains": {
    "Materials": [
        {
            "Attributes": attr("substrate"),
            "Permittivity": eps_r,
            "Permeability": 1.0,
            "Conductivity": sigma  # Replaced LossTan
        },
        {
            "Attributes": attr("air"),
            "Permittivity": 1.0,
            "Permeability": 1.0
        }
    ]
},

    "Boundaries": {
        "PEC": {
            "Attributes": attr("ground_plane") + attr("patch")
        },

        "LumpedPort": [
            {
                "Index": 1,
                "Attributes": attr("lumped_port_1"),
                "R": port_impedance,
                "Excitation": True,
                "Direction":  [0.0, 0.0, 1.0]
            },
            {
                "Index": 2,
                "Attributes": attr("lumped_port_2"),
                "R": port_impedance,
                "Excitation": False,              
                "Direction": [0.0, 0.0, 1.0]     
            }
        ],

        "Absorbing": {
            "Attributes": attr("farfield"),
            "Order": 1
        }
    },

    "Solver": {
    "Order": solver_order,
    "Device": "CPU",
    "Transient": {
      "Type": "GeneralizedAlpha",
      "Excitation": "ModulatedGaussian",
      "ExcitationFreq": freq, 
      "ExcitationWidth": 0.05, 
      "MaxTime": 1.0, 
      "TimeStep": 0.005, 
      "SaveStep": 10
    },
    "Linear": {
      "Type": "AMS",
      "KSPType": "CG",
      "Tol": 1.0e-8,
      "MaxIts": 100
    }
  }
}



script_dir = os.getcwd()
config_path = os.path.join(script_dir, output_file_transient)
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Palace config written to {config_path}")

Palace config written to c:\Users\loloc\Desktop\Epsilon\palace-course\lecture_3_waveports\l_antenna_transient.json


In [53]:
config = {
    "Problem": {
        "Type": "Driven",
        "Verbose": 2,
        "Output": "/work/results_driven/l_antenna/"
    },

    "Model": {
        "Mesh": f"/work/{filename}",
        "L0": 1.0,
        "Refinement": {}
    },

    "Domains": {
    "Materials": [
        {
            "Attributes": attr("substrate"),
            "Permittivity": eps_r,
            "Permeability": 1.0,
            "Conductivity": loss_tan 
        },
        {
            "Attributes": attr("air"),
            "Permittivity": 1.0,
            "Permeability": 1.0
        }
    ]
},

    "Boundaries": {
        "PEC": {
            "Attributes": attr("ground_plane") + attr("patch")
        },

        "LumpedPort": [
            {
                "Index": 1,
                "Attributes": attr("lumped_port_1"),
                "R": port_impedance,
                "Excitation": True,
                "Direction":  [0.0, 0.0, 1.0]
            },
            {
                "Index": 2,
                "Attributes": attr("lumped_port_2"),
                "R": port_impedance,
                "Excitation": False,              
                "Direction": [0.0, 0.0, 1.0]     
            }
        ],

        "Absorbing": {
            "Attributes": attr("farfield"),
            "Order": 1
        }
    },

  "Solver": {
    "Order": 2,
    "Device": "CPU",
    "Driven": {
      "MinFreq": 3.0,
      "MaxFreq": 3.5,
      "FreqStep": 0.1,
      "SaveStep": 1,
      "AdaptiveTol": 0.0001
    },
    "Linear": {
      "Type": "Default",
      "KSPType": "GMRES",
      "Tol": 1e-08,
      "MaxIts": 300,
      "MaxSize": 1000,
      "ComplexCoarseSolve": True
    }
  }
}



script_dir = os.getcwd()
config_path = os.path.join(script_dir, output_file_driven)
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)
print(f"Palace config written to {config_path}")

Palace config written to c:\Users\loloc\Desktop\Epsilon\palace-course\lecture_3_waveports\l_antenna_driven.json
